In [106]:
import datetime as dt
import re
import openpyxl
import numpy as np
import pandas as pd

DIR    = "../data"
F_HB   = f"{DIR}/Dive_depth___Hb_saturation_data_2-21-24.xlsx"
F_DIVE = f"{DIR}/Saturation_Profile_Groups_-_ID___Dive_Parameters.2-21-24xlsx.xlsx"
EPOCH  = dt.datetime(1970, 1, 1)

def norm_bird(s):
    """Chuẩn hóa mã sheet/chim: sửa lỗi gõ EPO -> EP0, xóa khoảng trắng thừa."""
    return str(s).strip().replace("EPO", "EP0")

def norm_dive_id(raw, bird):
    """Chuẩn hóa toàn bộ 160 dive_id về dạng 'EPxx n' (sửa EPO2 2, EP 17 15, EP1018, EP10  27, EP36->EP36A)."""
    s = re.sub(r"\s+", "", str(raw)).replace("EPO", "EP0")
    m = re.match(r"^EP(\d{2})A?(\d+)$", s)
    return f"{bird} {int(m.group(2))}" if m else " ".join(str(raw).split())

In [107]:
def tsec(v, mode):
    """Đổi mốc thời gian sang giây: mode='abs' (Unix epoch) hoặc 'tod' (giây trong ngày)."""
    if isinstance(v, dt.datetime):
        return (v - EPOCH).total_seconds() if mode == "abs" else v.hour*3600 + v.minute*60 + v.second
    if isinstance(v, dt.time):
        return v.hour*3600 + v.minute*60 + v.second
    if isinstance(v, dt.timedelta):
        return v.total_seconds()
    return None

def load_sheet(ws):
    """Đọc 1 sheet: trả về (chuỗi Hb đã nội suy depth, chuỗi depth 1 Hz gốc đã chuẩn hóa dấu)."""
    rows = list(ws.iter_rows(values_only=True))
    h = next(i for i, r in enumerate(rows) if r[0] in ("Date/Time", "Time"))
    hdr = [str(v).strip().lower() if v else None for v in rows[h]]
    data = rows[h+1:]
    po = [j for j, v in enumerate(hdr) if v == "po2"]
    dp = [j for j, v in enumerate(hdr) if v == "depth"]

    sat, dep = [], []
    for j in po:
        for r in data:
            if j+1 < len(r) and r[j-1] is not None and r[j] is not None and r[j+1] is not None:
                ts_val = r[j-1]
                mode_s = "abs" if isinstance(ts_val, dt.datetime) else "tod"
                d_val = ts_val.date() if isinstance(ts_val, dt.datetime) else None
                sat.append((tsec(ts_val, mode_s), float(r[j]), float(r[j+1]), d_val))
    for j in dp:
        for r in data:
            if j < len(r) and r[j-1] is not None and r[j] is not None:
                ts_val = r[j-1]
                mode_d = "abs" if isinstance(ts_val, dt.datetime) else "tod"
                dep.append((tsec(ts_val, mode_d), float(r[j])))

    s = pd.DataFrame(sat, columns=["t", "PO2", "Hb_pct", "date"]).sort_values("t").drop_duplicates("t")
    d = pd.DataFrame(dep, columns=["t", "depth_m"]).sort_values("t").drop_duplicates("t")

    # Chỉ đảo dấu sheet dùng quy ước âm là sâu (EP37), sau đó cắt trôi nền mặt nước < 0 về 0
    if d["depth_m"].min() < -10:
        d["depth_m"] = -d["depth_m"]
    d["depth_m"] = d["depth_m"].clip(lower=0.0)

    # Nội suy tuyến tính độ sâu lên thời điểm đo Hb (lọc bỏ nếu khoảng cách tới mốc đo độ sâu > 3s)
    if (d["t"] >= 86400).any():
        s["depth_m"] = np.interp(s.t, d.t, d.depth_m, left=np.nan, right=np.nan)
        k = np.searchsorted(d.t.values, s.t.values).clip(1, len(d)-1)
        gap = np.minimum(abs(d.t.values[k] - s.t.values), abs(d.t.values[k-1] - s.t.values))
    else:
        s_tod = s["t"] % 86400
        d_tod = d.assign(tod=d["t"] % 86400).sort_values("tod").drop_duplicates("tod")
        s["depth_m"] = np.interp(s_tod, d_tod.tod, d_tod.depth_m, left=np.nan, right=np.nan)
        k = np.searchsorted(d_tod.tod.values, s_tod.values).clip(1, len(d_tod)-1)
        gap = np.minimum(abs(d_tod.tod.values[k] - s_tod.values), abs(d_tod.tod.values[k-1] - s_tod.values))
    s.loc[gap > 3, "depth_m"] = np.nan

    d["tod"] = d["t"] % 86400
    d["date"] = pd.to_datetime(d["t"], unit="s").dt.date if (d["t"] >= 86400).any() else None
    return s, d

wb = openpyxl.load_workbook(F_HB, read_only=True)
parts, raw_1hz_map = [], {}
for ws in wb.worksheets:
    s, d = load_sheet(ws)
    b = norm_bird(ws.title.split()[0])
    site = "arterial" if "arterial" in ws.title.lower() else "venous"
    s["bird"], s["site"] = b, site
    parts.append(s)
    raw_1hz_map[(b, site)] = d

sat = pd.concat(parts, ignore_index=True)
sat["tod"] = sat.t % 86400

print(f"Tổng số mẫu Hb đã nạp: {len(sat):,}")
print(f"Số sheet: {len(wb.sheetnames)} | Số mã bird: {sat.bird.nunique()}")

Tổng số mẫu Hb đã nạp: 29,511
Số sheet: 15 | Số mã bird: 15


In [108]:
def sod(t):
    """Đổi time sang giây trong ngày."""
    return t.hour*3600 + t.minute*60 + t.second

ws_dive = openpyxl.load_workbook(F_DIVE, data_only=True).active
L = []
for r in ws_dive.iter_rows(min_row=5, values_only=True):
    if r[0]:
        b = norm_bird(r[0])
        L.append(("Type A venous", b, norm_dive_id(r[1], b), sod(r[4]), sod(r[5]), float(r[3])))
    if r[7]:
        b = norm_bird(r[7])
        L.append(("Type B venous", b, norm_dive_id(r[8], b), sod(r[11]), sod(r[12]), float(r[10])))
    if r[14]:
        b = norm_bird(r[14])
        L.append(("Arterial", b, norm_dive_id(r[15], b), sod(r[16]), sod(r[17]), float(r[19])))

D = pd.DataFrame(L, columns=["group", "bird", "dive_id", "start", "end", "max_depth"])
D["dur_min"] = (D.end - D.start) / 60

def assign_dive_date(row):
    """Gán ngày cho 2 chim có khung giờ chồng giữa 2 ngày đo (EP10 và EP48)."""
    num = int(row.dive_id.split()[1])
    if row.bird == "EP10":
        return dt.date(2007, 12, 3) if num <= 15 else dt.date(2007, 12, 4)
    if row.bird == "EP48":
        return dt.date(2005, 11, 26) if 7 <= num <= 12 else dt.date(2005, 11, 25)
    return None

D["date"] = D.apply(assign_dive_date, axis=1)

print(f"Tổng số lần lặn trong bảng: {len(D)}")
print(f"Số dive_id duy nhất: {D.dive_id.nunique()} (kỳ vọng: 160)")
print(D.groupby("group").size())

Tổng số lần lặn trong bảng: 160
Số dive_id duy nhất: 160 (kỳ vọng: 160)
group
Arterial         64
Type A venous    31
Type B venous    65
dtype: int64


In [109]:
sat["dive_id"] = None
sat["group"] = None
sat["t_in_dive_s"] = np.nan
sat["dive_dur_min"] = np.nan

for _, d in D.iterrows():
    expected_site = "arterial" if d.group == "Arterial" else "venous"
    m = (sat.bird == d.bird) & (sat.site == expected_site) & (sat.tod >= d.start) & (sat.tod <= d.end)
    if d.date is not None:
        m = m & (sat.date == d.date)
    sat.loc[m, ["dive_id", "group"]] = [d.dive_id, d.group]
    sat.loc[m, "t_in_dive_s"] = sat.loc[m, "tod"] - d.start
    sat.loc[m, "dive_dur_min"] = d.dur_min

sat["Hb"] = sat.Hb_pct / 100.0
sat = sat.sort_values(["bird", "t"]).reset_index(drop=True)

# --- TỔNG HỢP BẢNG KIỂM ĐỊNH 160 LẦN LẶN (D_clean) ---
raw_1hz_max = []
for _, d in D.iterrows():
    site = "arterial" if d.group == "Arterial" else "venous"
    dep_df = raw_1hz_map[(d.bird, site)]
    m = (dep_df["tod"] >= d.start) & (dep_df["tod"] <= d.end)
    if d.date is not None:
        m = m & (dep_df["date"] == d.date)
    raw_1hz_max.append(dep_df.loc[m, "depth_m"].max() if m.any() else np.nan)

g = sat.dropna(subset=["dive_id"]).groupby("dive_id")
audit = pd.DataFrame({
    "n_samples": g.size(),
    "obs_date": g.date.first(),
    "max_depth_obs": g.depth_m.max(),
    "coverage_pct": ((g.t.max() - g.t.min()) / (g.dive_dur_min.first() * 60) * 100).clip(upper=100.0),
}).reset_index()

D_clean = D.copy()
D_clean["max_depth_1hz"] = raw_1hz_max
D_clean = D_clean.merge(audit, on="dive_id", how="left")
D_clean["date"] = D_clean["date"].fillna(D_clean["obs_date"])
D_clean = D_clean.drop(columns=["obs_date"])
D_clean["n_samples"] = D_clean["n_samples"].fillna(0).astype(int)

# Cờ lần lặn hợp lệ (loại ca cắt cụt EP04 5 có 2 mẫu và 3 ca 0 mẫu)
D_clean["is_valid_dive"] = (D_clean["n_samples"] >= 8) & (D_clean["dur_min"] * 60 >= 100)

# Sửa điểm 1: Giữ nguyên max_depth chuẩn của bảng, chỉ sửa lỗi gõ EP10 7 (15.0m -> 24.0m từ 1 Hz gốc)
D_clean["max_depth_clean"] = D_clean["max_depth"]
D_clean.loc[D_clean.dive_id == "EP10 7", "max_depth_clean"] = 24.0

# --- CỜ CẤP DÒNG CHO CHUỖI THỜI GIAN ---
# (a) excluded_dive: CHỈ bằng True với dòng thuộc lần lặn bị loại (is_valid_dive = False ở D_clean, tức EP04 5).
#     Dòng ngoài mọi lần lặn (pha trước/sau lặn) luôn là False -> lọc ~excluded_dive không làm mất dữ liệu nền/hồi phục.
valid_ids = set(D_clean.loc[D_clean["is_valid_dive"], "dive_id"])
sat["excluded_dive"] = sat["dive_id"].notna() & ~sat["dive_id"].isin(valid_ids)

# (b) unlisted_dive: dòng nằm trong ±120s quanh một đợt lặn sâu (>10 m) KHÔNG có trong bảng 160 lần lặn.
#     Dòng sâu >10 m cách một cửa sổ lặn có trong bảng <= 60s chỉ là phần đuôi của lần lặn đó (giờ trong bảng ngắn hơn
#     thực tế), KHÔNG được kích hoạt cờ, để không gắn cờ nhầm các dòng phục hồi của lần lặn đã liệt kê.
EDGE_S, PAD_S = 60.0, 120.0

sat["gap_listed"] = np.inf   # khoảng cách (giây) tới cửa sổ lặn gần nhất trong bảng (cùng bird, site và ngày nếu có)
for _, d in D.iterrows():
    site_d = "arterial" if d.group == "Arterial" else "venous"
    m = (sat.bird == d.bird) & (sat.site == site_d)
    if d.date is not None:
        m = m & (sat.date == d.date)
    tod_m = sat.loc[m, "tod"].values
    gap = np.where(tod_m < d.start, d.start - tod_m, np.where(tod_m > d.end, tod_m - d.end, 0.0))
    sat.loc[m, "gap_listed"] = np.minimum(sat.loc[m, "gap_listed"].values, gap)

deep_all      = sat[sat["dive_id"].isna() & (sat["depth_m"] > 10)]
deep_edge     = deep_all[deep_all["gap_listed"] <= EDGE_S]
deep_unlisted = deep_all[deep_all["gap_listed"] > EDGE_S]

sat["unlisted_dive"] = False
for (b, site), grp in deep_unlisted.groupby(["bird", "site"]):
    t_deep = np.sort(grp["t"].values)
    m_bs = (sat["bird"] == b) & (sat["site"] == site) & sat["dive_id"].isna()
    t_all = sat.loc[m_bs, "t"].values
    idx = np.searchsorted(t_deep, t_all).clip(0, len(t_deep) - 1)
    idx_prev = (idx - 1).clip(0, len(t_deep) - 1)
    min_dist = np.minimum(np.abs(t_all - t_deep[idx]), np.abs(t_all - t_deep[idx_prev]))
    sat.loc[m_bs, "unlisted_dive"] = (min_dist <= PAD_S)

# --- KIỂM ĐỊNH ĐỘC LẬP BẰNG ASSERT ---
BASE_COLS = [
    "bird", "site", "group", "dive_id",
    "t", "tod", "date", "t_in_dive_s", "dive_dur_min",
    "PO2", "Hb_pct", "Hb", "depth_m"
]
FLAG_COLS = ["excluded_dive", "unlisted_dive"]

assert len(sat) == 29511, f"Sai tổng số dòng: {len(sat)}"
assert list(sat[BASE_COLS + FLAG_COLS].columns) == BASE_COLS + FLAG_COLS
assert sat.duplicated(["bird", "site", "t"]).sum() == 0, "Có bản ghi trùng (bird, site, t)!"
assert D_clean.dive_id.nunique() == 160, "Thiếu mã dive_id!"
assert (sat["depth_m"].dropna() >= 0).all(), "Còn độ sâu âm chưa clip về 0!"
assert np.isclose(sat[sat.bird == "EP37"]["depth_m"].max(), 155.0), "EP37 chưa đảo dấu đúng!"
assert not (sat["dive_id"].notna() & sat["unlisted_dive"]).any(), "unlisted_dive đè lên lần lặn đã gán!"
assert not (sat["dive_id"].isna() & sat["excluded_dive"]).any(), "excluded_dive gắn nhầm dòng ngoài lần lặn!"
assert sat["excluded_dive"].sum() == 2, "excluded_dive phải đúng 2 dòng (EP04 5)!"
assert (sat.loc[sat["dive_id"].notna(), "gap_listed"] == 0).all(), "gap_listed sai: dòng trong lần lặn phải có khoảng cách 0!"
assert (deep_unlisted["gap_listed"] > EDGE_S).all(), "deep_unlisted còn dòng sát cửa sổ lặn trong bảng!"

in_dive = sat.dropna(subset=["dive_id"])
assert ((in_dive["t_in_dive_s"] >= 0) & (in_dive["t_in_dive_s"] <= in_dive["dive_dur_min"] * 60)).all()

hill_pct = 100.0 * (sat["PO2"] ** 2.93) / (28.2 ** 2.93 + sat["PO2"] ** 2.93)
assert (sat["Hb_pct"] - hill_pct).abs().max() < 0.1, "Lệch công thức Hill vượt ngưỡng 0.1%!"

# Sửa điểm 3: Siết ngưỡng kiểm tra độ sâu trực tiếp với dữ liệu 1 Hz gốc (<= 4.5 m)
valid_dives = D_clean[D_clean["is_valid_dive"]]
assert (valid_dives["max_depth_clean"] - valid_dives["max_depth_1hz"]).abs().max() <= 4.5

print("[PASS] Toàn bộ kiểm định độc lập đạt chuẩn (bao gồm đối chiếu độ sâu 1 Hz <= 4.5 m)!")
print(f"- Dòng trong lần lặn hợp lệ: {(sat['dive_id'].notna() & ~sat['excluded_dive']).sum():,} dòng")
print(f"- Dòng thuộc lần lặn bị loại (excluded_dive=True, EP04 5): {sat['excluded_dive'].sum()} dòng")
print(f"- Dòng sâu >10m ngoài bảng: {len(deep_all)} dòng, trong đó {len(deep_edge)} dòng chỉ là phần đuôi (<= {EDGE_S:.0f}s) của lần lặn đã liệt kê")
print(f"- Dòng thuộc đợt lặn ngoài bảng (unlisted_dive=True, ±{PAD_S:.0f}s quanh {len(deep_unlisted)} dòng sâu): {sat['unlisted_dive'].sum():,} dòng")


[PASS] Toàn bộ kiểm định độc lập đạt chuẩn (bao gồm đối chiếu độ sâu 1 Hz <= 4.5 m)!
- Dòng trong lần lặn hợp lệ: 5,927 dòng
- Dòng thuộc lần lặn bị loại (excluded_dive=True, EP04 5): 2 dòng
- Dòng sâu >10m ngoài bảng: 368 dòng, trong đó 18 dòng chỉ là phần đuôi (<= 60s) của lần lặn đã liệt kê
- Dòng thuộc đợt lặn ngoài bảng (unlisted_dive=True, ±120s quanh 350 dòng sâu): 974 dòng


In [110]:
cols = [
    "bird", "site", "group", "dive_id",
    "t", "tod", "date", "t_in_dive_s", "dive_dur_min",
    "PO2", "Hb_pct", "Hb", "depth_m",
    "excluded_dive", "unlisted_dive"
]

csv_path   = f"{DIR}/penguin_merged.csv"
dives_path = f"{DIR}/penguin_dives.csv"

sat[cols].to_csv(csv_path, index=False)
D_clean.to_csv(dives_path, index=False)

print(f"Đã xuất chuỗi thời gian sạch: {csv_path} ({len(sat):,} dòng, {len(cols)} cột)")
print(f"Đã xuất bảng danh mục 160 lần lặn: {dives_path} ({len(D_clean)} dòng)")

Đã xuất chuỗi thời gian sạch: ../data/penguin_merged.csv (29,511 dòng, 15 cột)
Đã xuất bảng danh mục 160 lần lặn: ../data/penguin_dives.csv (160 dòng)


In [111]:
xls_raw = pd.ExcelFile(F_HB)
SHEET_MAP = {
    (norm_bird(s.split()[0]), "arterial" if "arterial" in s.lower() else "venous"): s
    for s in xls_raw.sheet_names
}
df_merged_chk = pd.read_csv(f"{DIR}/penguin_merged.csv")
df_dives_chk  = pd.read_csv(f"{DIR}/penguin_dives.csv")

def fmt_tod(sec):
    return str(dt.timedelta(seconds=int(sec)))

def verify_bird(bird, site):
    """Đối chiếu độc lập giữa Excel gốc (đọc qua pandas) và file CSV đã merge của 1 cá thể."""
    sname = SHEET_MAP[(bird, site)]
    raw_df = pd.read_excel(xls_raw, sheet_name=sname, header=4)

    # Đọc độc lập tất cả cụm cột PO2 / Hb saturation và Depth từ sheet gốc
    po2_cols = [c for c in raw_df.columns if str(c).strip().lower().startswith("po2")]
    dep_cols = [c for c in raw_df.columns if str(c).strip().lower().startswith("depth")]

    raw_po2, raw_hb, raw_dep = [], [], []
    for pc in po2_cols:
        idx = list(raw_df.columns).index(pc)
        tc, hc = raw_df.columns[idx - 1], raw_df.columns[idx + 1]
        sub = raw_df[[tc, pc, hc]].dropna()
        raw_po2.extend(pd.to_numeric(sub[pc], errors="coerce").tolist())
        raw_hb.extend(pd.to_numeric(sub[hc], errors="coerce").tolist())

    for dc in dep_cols:
        idx = list(raw_df.columns).index(dc)
        tc = raw_df.columns[idx - 1]
        sub = raw_df[[tc, dc]].dropna()
        raw_dep.extend(pd.to_numeric(sub[dc], errors="coerce").tolist())

    raw_dep = np.array(raw_dep, dtype=float)
    if raw_dep.min() < -10:
        raw_dep = -raw_dep
    raw_dep = np.clip(raw_dep, 0.0, None)

    sub_csv = df_merged_chk[(df_merged_chk["bird"] == bird) & (df_merged_chk["site"] == site)].copy()
    sub_d   = df_dives_chk[(df_dives_chk["bird"] == bird) & ((df_dives_chk["group"] == "Arterial") if site == "arterial" else (df_dives_chk["group"] != "Arterial"))].copy()

    # Kiểm tra độc lập công thức Hill & các cờ chất lượng
    hill_err = (sub_csv["Hb_pct"] - 100.0 * (sub_csv["PO2"] ** 2.93) / (28.2 ** 2.93 + sub_csv["PO2"] ** 2.93)).abs().max()
    unlisted_deep = len(sub_csv[sub_csv["dive_id"].isna() & (sub_csv["depth_m"] > 10)])
    unlisted_flag = int(sub_csv["unlisted_dive"].sum())

    print(f"================ KIỂM TRA: {bird} ({site.upper()}) | Sheet gốc: '{sname}' ================")
    print(f"- Sai số công thức Hill max: {hill_err:.4f}% | Lặn ngoài bảng >10m: {unlisted_deep} dòng (cờ ±120s: {unlisted_flag} dòng)")

    # 1. Bảng so sánh tổng quan giữa Data gốc (Excel) và Data đã merge (CSV)
    comp = pd.DataFrame({
        "Chỉ tiêu": [
            "Số mẫu Hb / PO2",
            "PO2 min (mmHg)",
            "PO2 max (mmHg)",
            "Hb_pct min (%)",
            "Hb_pct max (%)",
            "Độ sâu max (m)",
            "Số lần lặn khớp bảng",
        ],
        "Data gốc (Excel)": [
            f"{len(raw_po2):,}",
            f"{min(raw_po2):.1f}",
            f"{max(raw_po2):.1f}",
            f"{min(raw_hb):.4f}",
            f"{max(raw_hb):.4f}",
            f"{raw_dep.max():.1f} (1 Hz)",
            f"{len(sub_d)} lần",
        ],
        "Data đã merge (CSV)": [
            f"{len(sub_csv):,}",
            f"{sub_csv['PO2'].min():.1f}",
            f"{sub_csv['PO2'].max():.1f}",
            f"{sub_csv['Hb_pct'].min():.4f}",
            f"{sub_csv['Hb_pct'].max():.4f}",
            f"{sub_csv['depth_m'].max():.1f} (mốc Hb)",
            f"{(sub_d['n_samples'] > 0).sum()}/{len(sub_d)} lần ({int(sub_csv['dive_id'].notna().sum())} mẫu)",
        ],
        "Trạng thái": [
            "KHỚP" if len(raw_po2) == len(sub_csv) else "LỆCH",
            "KHỚP" if np.isclose(min(raw_po2), sub_csv["PO2"].min()) else "LỆCH",
            "KHỚP" if np.isclose(max(raw_po2), sub_csv["PO2"].max()) else "LỆCH",
            "KHỚP" if np.isclose(min(raw_hb), sub_csv["Hb_pct"].min()) else "LỆCH",
            "KHỚP" if np.isclose(max(raw_hb), sub_csv["Hb_pct"].max()) else "LỆCH",
            "KHỚP" if sub_csv["depth_m"].max() <= raw_dep.max() + 1e-6 else "LỆCH",
            "KHỚP" if (sub_d["n_samples"] > 0).sum() == len(sub_d) else f"THIẾU {(sub_d['n_samples'] == 0).sum()} CA GỐC",
        ]
    })
    print(comp.to_string(index=False))

    # 2. Bảng đối chiếu chi tiết từng lần lặn của cá thể
    dive_stats = []
    for _, d in sub_d.iterrows():
        dive_stats.append({
            "dive_id": d["dive_id"],
            "group": d["group"],
            "date": d["date"],
            "bắt_đầu": fmt_tod(d["start"]),
            "kết_thúc": fmt_tod(d["end"]),
            "dur_min": round(d["dur_min"], 2),
            "số_mẫu": int(d["n_samples"]),
            "depth_bảng": d["max_depth"],
            "depth_1Hz": round(d["max_depth_1hz"], 1) if pd.notna(d["max_depth_1hz"]) else np.nan,
            "depth_clean": round(d["max_depth_clean"], 1) if pd.notna(d["max_depth_clean"]) else np.nan,
            "lệch_1Hz": round(abs(d["max_depth_clean"] - d["max_depth_1hz"]), 1) if pd.notna(d["max_depth_1hz"]) else np.nan,
            "hợp_lệ": bool(d["is_valid_dive"]),
        })
    print(f"\n--- Chi tiết {len(sub_d)} lần lặn của {bird} ({site}) ---")
    print(pd.DataFrame(dive_stats).to_string(index=False))

In [112]:
# Cell 7: EP01 (Arterial - 13 lần lặn, 2.878 mẫu)
verify_bird("EP01", "arterial")

================ KIỂM TRA: EP01 (ARTERIAL) | Sheet gốc: 'EP01 arterial' ================
- Sai số công thức Hill max: 0.0352% | Lặn ngoài bảng >10m: 40 dòng (cờ ±120s: 226 dòng)
            Chỉ tiêu Data gốc (Excel)  Data đã merge (CSV) Trạng thái
     Số mẫu Hb / PO2            2,878                2,878       KHỚP
      PO2 min (mmHg)             28.0                 28.0       KHỚP
      PO2 max (mmHg)            218.0                218.0       KHỚP
      Hb_pct min (%)          49.4730              49.4730       KHỚP
      Hb_pct max (%)          99.7487              99.7487       KHỚP
      Độ sâu max (m)      91.5 (1 Hz)        91.0 (mốc Hb)       KHỚP
Số lần lặn khớp bảng           13 lần 13/13 lần (1156 mẫu)       KHỚP

--- Chi tiết 13 lần lặn của EP01 (arterial) ---
dive_id    group       date  bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
 EP01 1 Arterial 2008-11-26  8:24:53  8:31:47     6.90      83        55.0       56.0         55

In [113]:
# Cell 8: EP02 (Arterial - 1 lần lặn, 17 mẫu - đã chuẩn hóa từ 'EPO2 arterial')
verify_bird("EP02", "arterial")

================ KIỂM TRA: EP02 (ARTERIAL) | Sheet gốc: 'EPO2 arterial' ================
- Sai số công thức Hill max: 0.0352% | Lặn ngoài bảng >10m: 0 dòng (cờ ±120s: 0 dòng)
            Chỉ tiêu Data gốc (Excel) Data đã merge (CSV) Trạng thái
     Số mẫu Hb / PO2               17                  17       KHỚP
      PO2 min (mmHg)             35.0                35.0       KHỚP
      PO2 max (mmHg)            128.0               128.0       KHỚP
      Hb_pct min (%)          65.2901             65.2901       KHỚP
      Hb_pct max (%)          98.8176             98.8176       KHỚP
      Độ sâu max (m)      27.5 (1 Hz)       25.5 (mốc Hb)       KHỚP
Số lần lặn khớp bảng            1 lần    1/1 lần (13 mẫu)       KHỚP

--- Chi tiết 1 lần lặn của EP02 (arterial) ---
dive_id    group       date bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
 EP02 2 Arterial 2003-11-23 9:02:06  9:05:11     3.08      13        24.0       27.5         24.0       3.5  

In [114]:
# Cell 9: EP04 (Venous - 5 lần lặn, 166 mẫu - kiểm tra ca EP04 5 chỉ có 2 mẫu)
verify_bird("EP04", "venous")

================ KIỂM TRA: EP04 (VENOUS) | Sheet gốc: 'EP04' ================
- Sai số công thức Hill max: 0.0352% | Lặn ngoài bảng >10m: 0 dòng (cờ ±120s: 0 dòng)
            Chỉ tiêu Data gốc (Excel) Data đã merge (CSV) Trạng thái
     Số mẫu Hb / PO2              166                 166       KHỚP
      PO2 min (mmHg)             33.0                33.0       KHỚP
      PO2 max (mmHg)            103.0               103.0       KHỚP
      Hb_pct min (%)          61.2932             61.2932       KHỚP
      Hb_pct max (%)          97.7903             97.7903       KHỚP
      Độ sâu max (m)      21.0 (1 Hz)       20.0 (mốc Hb)       KHỚP
Số lần lặn khớp bảng            5 lần    5/5 lần (66 mẫu)       KHỚP

--- Chi tiết 5 lần lặn của EP04 (venous) ---
dive_id         group       date bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
 EP04 1 Type A venous 2003-11-14 9:03:34  9:07:17     3.72      15        16.5       16.5         16.5       0.0    T

In [115]:
# Cell 10: EP05 (Venous - 11 lần lặn, 2.389 mẫu)
verify_bird("EP05", "venous")

================ KIỂM TRA: EP05 (VENOUS) | Sheet gốc: 'EP05' ================
- Sai số công thức Hill max: 0.0352% | Lặn ngoài bảng >10m: 0 dòng (cờ ±120s: 0 dòng)
            Chỉ tiêu Data gốc (Excel) Data đã merge (CSV) Trạng thái
     Số mẫu Hb / PO2            2,389               2,389       KHỚP
      PO2 min (mmHg)              8.0                 8.0       KHỚP
      PO2 max (mmHg)            104.0               104.0       KHỚP
      Hb_pct min (%)           2.4446              2.4446       KHỚP
      Hb_pct max (%)          97.8506             97.8506       KHỚP
      Độ sâu max (m)      61.0 (1 Hz)       61.0 (mốc Hb)       KHỚP
Số lần lặn khớp bảng           11 lần 11/11 lần (659 mẫu)       KHỚP

--- Chi tiết 11 lần lặn của EP05 (venous) ---
dive_id         group       date  bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
 EP05 1 Type B venous 2003-11-17 20:41:18 20:44:25     3.12      38        13.5       13.5         13.5       0.0  

In [116]:
# Cell 11: EP07 (Venous - 10 lần lặn, 3.991 mẫu - kiểm tra EP07 9 & EP07 10 không có mẫu Hb)
verify_bird("EP07", "venous")

================ KIỂM TRA: EP07 (VENOUS) | Sheet gốc: 'EP07' ================
- Sai số công thức Hill max: 0.0352% | Lặn ngoài bảng >10m: 0 dòng (cờ ±120s: 0 dòng)
            Chỉ tiêu Data gốc (Excel) Data đã merge (CSV)     Trạng thái
     Số mẫu Hb / PO2            3,991               3,991           KHỚP
      PO2 min (mmHg)             15.0                15.0           KHỚP
      PO2 max (mmHg)            183.0               183.0           KHỚP
      Hb_pct min (%)          13.6192             13.6192           KHỚP
      Hb_pct max (%)          99.5813             99.5813           KHỚP
      Độ sâu max (m)      39.5 (1 Hz)       35.5 (mốc Hb)           KHỚP
Số lần lặn khớp bảng           10 lần  8/10 lần (162 mẫu) THIẾU 2 CA GỐC

--- Chi tiết 10 lần lặn của EP07 (venous) ---
dive_id         group       date  bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
 EP07 1 Type A venous 2003-11-12 19:11:58 19:14:49     2.85      11        17.0    

In [117]:
# Cell 12: EP08 (Arterial - 12 lần lặn, 5.173 mẫu - 2 phiên đo trong cùng ngày 04/12/2007)
verify_bird("EP08", "arterial")

================ KIỂM TRA: EP08 (ARTERIAL) | Sheet gốc: 'EP08 arterial' ================
- Sai số công thức Hill max: 0.0352% | Lặn ngoài bảng >10m: 0 dòng (cờ ±120s: 0 dòng)
            Chỉ tiêu Data gốc (Excel) Data đã merge (CSV) Trạng thái
     Số mẫu Hb / PO2            5,173               5,173       KHỚP
      PO2 min (mmHg)             29.0                29.0       KHỚP
      PO2 max (mmHg)            149.0               149.0       KHỚP
      Hb_pct min (%)          52.0387             52.0387       KHỚP
      Hb_pct max (%)          99.2387             99.2387       KHỚP
      Độ sâu max (m)      28.0 (1 Hz)       27.5 (mốc Hb)       KHỚP
Số lần lặn khớp bảng           12 lần 12/12 lần (720 mẫu)       KHỚP

--- Chi tiết 12 lần lặn của EP08 (arterial) ---
dive_id    group       date  bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
 EP08 1 Arterial 2007-12-04  8:59:07  9:02:05     2.97      36        11.0       11.0         11.0       0.

In [118]:
# Cell 13: EP10 (Arterial - 32 lần lặn, 4.006 mẫu - tách 2 ngày 03/12 và 04/12/2007)
verify_bird("EP10", "arterial")

================ KIỂM TRA: EP10 (ARTERIAL) | Sheet gốc: 'EP10 arterial' ================
- Sai số công thức Hill max: 0.0352% | Lặn ngoài bảng >10m: 1 dòng (cờ ±120s: 17 dòng)
            Chỉ tiêu Data gốc (Excel) Data đã merge (CSV) Trạng thái
     Số mẫu Hb / PO2            4,006               4,006       KHỚP
      PO2 min (mmHg)             31.0                31.0       KHỚP
      PO2 max (mmHg)            161.0               161.0       KHỚP
      Hb_pct min (%)          56.8743             56.8743       KHỚP
      Hb_pct max (%)          99.3921             99.3921       KHỚP
      Độ sâu max (m)      29.5 (1 Hz)       29.0 (mốc Hb)       KHỚP
Số lần lặn khớp bảng           32 lần 32/32 lần (666 mẫu)       KHỚP

--- Chi tiết 32 lần lặn của EP10 (arterial) ---
dive_id    group       date  bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
 EP10 1 Arterial 2007-12-03  8:27:37  8:33:02     5.42      21        15.5       16.0         15.5       0

In [119]:
# Cell 14: EP17 (Venous - 14 lần lặn, 3.245 mẫu - kiểm tra mã 'EP 17 15')
verify_bird("EP17", "venous")

================ KIỂM TRA: EP17 (VENOUS) | Sheet gốc: 'EP17' ================
- Sai số công thức Hill max: 0.0352% | Lặn ngoài bảng >10m: 148 dòng (cờ ±120s: 316 dòng)
            Chỉ tiêu Data gốc (Excel) Data đã merge (CSV) Trạng thái
     Số mẫu Hb / PO2            3,245               3,245       KHỚP
      PO2 min (mmHg)             14.0                14.0       KHỚP
      PO2 max (mmHg)             74.0                74.0       KHỚP
      Hb_pct min (%)          11.4138             11.4138       KHỚP
      Hb_pct max (%)          94.3878             94.3878       KHỚP
      Độ sâu max (m)      48.5 (1 Hz)       48.0 (mốc Hb)       KHỚP
Số lần lặn khớp bảng           14 lần 14/14 lần (337 mẫu)       KHỚP

--- Chi tiết 14 lần lặn của EP17 (venous) ---
dive_id         group       date  bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
 EP17 1 Type B venous 2004-12-02  8:52:04  8:59:35     7.52      31        31.5       31.5         31.5       0

In [120]:
# Cell 15: EP19 (Venous - 19 lần lặn, 2.221 mẫu)
verify_bird("EP19", "venous")

================ KIỂM TRA: EP19 (VENOUS) | Sheet gốc: 'EP19' ================
- Sai số công thức Hill max: 0.0352% | Lặn ngoài bảng >10m: 73 dòng (cờ ±120s: 156 dòng)
            Chỉ tiêu Data gốc (Excel) Data đã merge (CSV) Trạng thái
     Số mẫu Hb / PO2            2,221               2,221       KHỚP
      PO2 min (mmHg)              6.0                 6.0       KHỚP
      PO2 max (mmHg)            111.0               111.0       KHỚP
      Hb_pct min (%)           1.0684              1.0684       KHỚP
      Hb_pct max (%)          98.2169             98.2169       KHỚP
      Độ sâu max (m)      67.5 (1 Hz)       67.5 (mốc Hb)       KHỚP
Số lần lặn khớp bảng           19 lần 19/19 lần (828 mẫu)       KHỚP

--- Chi tiết 19 lần lặn của EP19 (venous) ---
dive_id         group       date  bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
 EP19 2 Type B venous 2004-11-29  9:01:05  9:10:01     8.93      36        45.0       41.5         45.0       3.

In [121]:
# Cell 16: EP20 (Venous - 3 lần lặn, 213 mẫu)
verify_bird("EP20", "venous")

================ KIỂM TRA: EP20 (VENOUS) | Sheet gốc: 'EP20' ================
- Sai số công thức Hill max: 0.0352% | Lặn ngoài bảng >10m: 4 dòng (cờ ±120s: 17 dòng)
            Chỉ tiêu Data gốc (Excel) Data đã merge (CSV) Trạng thái
     Số mẫu Hb / PO2              213                 213       KHỚP
      PO2 min (mmHg)             11.0                11.0       KHỚP
      PO2 max (mmHg)             59.0                59.0       KHỚP
      Hb_pct min (%)           5.9818              5.9818       KHỚP
      Hb_pct max (%)          89.6570             89.6570       KHỚP
      Độ sâu max (m)      83.5 (1 Hz)       81.5 (mốc Hb)       KHỚP
Số lần lặn khớp bảng            3 lần    3/3 lần (86 mẫu)       KHỚP

--- Chi tiết 3 lần lặn của EP20 (venous) ---
dive_id         group       date  bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
 EP20 1 Type A venous 2004-11-11 14:41:00 14:43:35     2.58      11        37.0       37.5         37.0       0.5  

In [122]:
# Cell 17: EP36 (Venous - 5 lần lặn, 601 mẫu)
verify_bird("EP36", "venous")

================ KIỂM TRA: EP36 (VENOUS) | Sheet gốc: 'EP36' ================
- Sai số công thức Hill max: 0.0352% | Lặn ngoài bảng >10m: 1 dòng (cờ ±120s: 0 dòng)
            Chỉ tiêu Data gốc (Excel) Data đã merge (CSV) Trạng thái
     Số mẫu Hb / PO2              601                 601       KHỚP
      PO2 min (mmHg)              1.0                 1.0       KHỚP
      PO2 max (mmHg)             46.0                46.0       KHỚP
      Hb_pct min (%)           0.0057              0.0057       KHỚP
      Hb_pct max (%)          80.7126             80.7126       KHỚP
      Độ sâu max (m)      93.0 (1 Hz)       92.0 (mốc Hb)       KHỚP
Số lần lặn khớp bảng            5 lần   5/5 lần (210 mẫu)       KHỚP

--- Chi tiết 5 lần lặn của EP36 (venous) ---
dive_id         group       date  bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
 EP36 1 Type B venous 2005-12-01 18:22:28 18:31:27     8.98      36        44.0       44.0         44.0       0.0   

In [123]:
# Cell 18: EP36A (Arterial - 6 lần lặn, 145 mẫu - đã sửa 'EP36 n' -> 'EP36A n', kiểm tra EP36A 8)
verify_bird("EP36A", "arterial")

================ KIỂM TRA: EP36A (ARTERIAL) | Sheet gốc: 'EP36A arterial' ================
- Sai số công thức Hill max: 0.0326% | Lặn ngoài bảng >10m: 5 dòng (cờ ±120s: 19 dòng)
            Chỉ tiêu Data gốc (Excel) Data đã merge (CSV)     Trạng thái
     Số mẫu Hb / PO2              145                 145           KHỚP
      PO2 min (mmHg)             55.0                55.0           KHỚP
      PO2 max (mmHg)            265.0               265.0           KHỚP
      Hb_pct min (%)          87.5912             87.5912           KHỚP
      Hb_pct max (%)          99.8579             99.8579           KHỚP
      Độ sâu max (m)      60.0 (1 Hz)       60.0 (mốc Hb)           KHỚP
Số lần lặn khớp bảng            6 lần    5/6 lần (87 mẫu) THIẾU 1 CA GỐC

--- Chi tiết 6 lần lặn của EP36A (arterial) ---
dive_id    group  date  bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
EP36A 1 Arterial   NaN  9:19:08  9:23:04     3.93      15        25.0       2

In [124]:
# Cell 19: EP37 (Venous - 9 lần lặn, 680 mẫu - kiểm tra đảo dấu độ sâu từ -155m -> +155m)
verify_bird("EP37", "venous")

================ KIỂM TRA: EP37 (VENOUS) | Sheet gốc: 'EP37' ================
- Sai số công thức Hill max: 0.0518% | Lặn ngoài bảng >10m: 6 dòng (cờ ±120s: 35 dòng)
            Chỉ tiêu Data gốc (Excel) Data đã merge (CSV) Trạng thái
     Số mẫu Hb / PO2              680                 680       KHỚP
      PO2 min (mmHg)             26.0                26.0       KHỚP
      PO2 max (mmHg)             69.0                69.0       KHỚP
      Hb_pct min (%)          44.0563             44.0563       KHỚP
      Hb_pct max (%)          93.1936             93.1936       KHỚP
      Độ sâu max (m)     155.0 (1 Hz)      155.0 (mốc Hb)       KHỚP
Số lần lặn khớp bảng            9 lần   9/9 lần (182 mẫu)       KHỚP

--- Chi tiết 9 lần lặn của EP37 (venous) ---
dive_id         group  date  bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
 EP37 1 Type A venous   NaN 17:44:16 17:46:17     2.02       8        10.0       13.2         10.0       3.2    True
 EP

In [125]:
# Cell 20: EP40 (Venous - 2 lần lặn, 202 mẫu)
verify_bird("EP40", "venous")

================ KIỂM TRA: EP40 (VENOUS) | Sheet gốc: 'EP40' ================
- Sai số công thức Hill max: 0.0352% | Lặn ngoài bảng >10m: 3 dòng (cờ ±120s: 0 dòng)
            Chỉ tiêu Data gốc (Excel) Data đã merge (CSV) Trạng thái
     Số mẫu Hb / PO2              202                 202       KHỚP
      PO2 min (mmHg)             11.0                11.0       KHỚP
      PO2 max (mmHg)             46.0                46.0       KHỚP
      Hb_pct min (%)           5.9818              5.9818       KHỚP
      Hb_pct max (%)          80.7126             80.7126       KHỚP
      Độ sâu max (m)      76.0 (1 Hz)       74.0 (mốc Hb)       KHỚP
Số lần lặn khớp bảng            2 lần    2/2 lần (90 mẫu)       KHỚP

--- Chi tiết 2 lần lặn của EP40 (venous) ---
dive_id         group       date  bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
 EP40 2 Type B venous 2005-12-01 18:24:22 18:34:55    10.55      42        75.5       76.0         75.5       0.5   

In [126]:
# Cell 21: EP48 (Venous - 18 lần lặn, 3.584 mẫu - tách 2 ngày 25/11 và 26/11/2005)
verify_bird("EP48", "venous")

================ KIỂM TRA: EP48 (VENOUS) | Sheet gốc: 'EP48' ================
- Sai số công thức Hill max: 0.0352% | Lặn ngoài bảng >10m: 87 dòng (cờ ±120s: 188 dòng)
            Chỉ tiêu Data gốc (Excel) Data đã merge (CSV) Trạng thái
     Số mẫu Hb / PO2            3,584               3,584       KHỚP
      PO2 min (mmHg)              3.0                 3.0       KHỚP
      PO2 max (mmHg)             50.0                50.0       KHỚP
      Hb_pct min (%)           0.1419              0.1419       KHỚP
      Hb_pct max (%)          84.2295             84.2295       KHỚP
      Độ sâu max (m)      71.5 (1 Hz)       71.5 (mốc Hb)       KHỚP
Số lần lặn khớp bảng           18 lần 18/18 lần (667 mẫu)       KHỚP

--- Chi tiết 18 lần lặn của EP48 (venous) ---
dive_id         group       date  bắt_đầu kết_thúc  dur_min  số_mẫu  depth_bảng  depth_1Hz  depth_clean  lệch_1Hz  hợp_lệ
 EP48 1 Type B venous 2005-11-25  9:54:32 10:04:18     9.77      39        52.5       53.0         52.5       0.